# 1차 결과 진단 (DIGNOSIS_1.md) — 추론만, 재학습 없음

기존 체크포인트(`runs/nad_N2_S01_s1`, `nad_N1-otf_otf_s1`, `nad_N1-pre_S01_s1`)를 **읽기만** 합니다. 결과는 새 폴더 `New_Architecture_dev/diag/DIAG_01/`에 씁니다.

| 셀 | 하는 일 | 결과 |
|---|---|---|
| 3 | val/test 오디오 캐시(코랩 로컬, 약 1.2GB)와 정답 csv 준비 | 로컬 |
| 4 | 1~4번: N2 후처리 4조합, 영상 원래/0/섞기, 앞/뒤 정확도, onscreen 점검 | `diag/DIAG_01/n2.json` |
| 5 | 3번 N1-otf 앞/뒤, 5번 N1-pre vs N1-otf 부트스트랩(1,000번) | `diag/DIAG_01/n1.json` |
| 6 | 표 만들기 | `diag/DIAG_01/tables.md` |

- 영상 0: 표준화 뒤 0이 되도록(원값 = train 평균) 넣습니다. (C) 열 토큰과 (D) 회전 특징이 같은 입력을 쓰므로 둘 다 막히고, 후처리 위치 보정도 꺼집니다(영상 점수 없음).
- 영상 섞기: 베이스라인과 같은 규칙(시드 1000+seed, 다른 클립의 영상). 후처리 보정도 섞인 영상 점수를 씁니다.
- 영상/앞뒤/onscreen 표의 후처리는 1번에서 **val** F_ext가 가장 높은 조합을 씁니다.
- 셀 4, 5는 끝나면 json을 저장하고, 다시 실행하면 건너뜁니다(다시 하려면 json을 지우세요). 시간은 측정하지 않았습니다(셀 4는 수십 분 예상).

## 1. 드라이브 연결, 패키지

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks/ASR_2026-2"   #@param {type:"string"}
NAD_DIR = f"{DRIVE_ROOT}/New_Architecture_dev"
import os
os.chdir(NAD_DIR)
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
!pip install -q librosa soundfile pyyaml tqdm
!nvidia-smi -L
!df -h /content | tail -1

## 2. 설정 칸 (학습 때와 같은 값)

In [ ]:
seed = 1                   #@param {type:"integer"}
train_stages = "0,1"       #@param ["0,1", "0"]
batch_size = 32            #@param {type:"integer"}
num_workers = 6            #@param {type:"integer"}
D = ' '.join(f'--set {k}="{v}"' for k, v in dict(drive_root=DRIVE_ROOT, seed=seed, train_stages=train_stages,
                                                   batch_size=batch_size, num_workers=num_workers).items())
print(D)

## 3. val/test 준비 (train 캐시는 만들지 않음)

In [ ]:
!python -B diag_run.py prepare {D}

## 4. N2 진단 (1~4번)

In [ ]:
!python -B diag_run.py n2 {D}

## 5. N1-otf 앞/뒤, N1-pre vs N1-otf 부트스트랩 (3, 5번)

In [ ]:
!python -B diag_run.py n1 {D}

## 6. 표

In [ ]:
!python -B diag_run.py tables {D}
from IPython.display import Markdown, display
display(Markdown(open(f"{NAD_DIR}/diag/DIAG_01/tables.md", encoding='utf8').read()))

## 7. W1 모델 진단 (MODEL_DEV_2.md, `nad_N1w_otf_s1`, `nad_N2w_S01_s1`)

결과는 새 폴더 `diag/W1_N2w/`에 씁니다. `DIAG_01`은 건드리지 않습니다. 셀 3(준비)을 먼저 실행하세요.

| 셀 | 하는 일 | 결과 |
|---|---|---|
| 7-1 | N2w 1~4번 (후처리 조합 a 고정, 기존 N2 `postproc.json`) | `n2.json` (이미 있으면 건너뜀) |
| 7-2 | N1w 앞/뒤(항상 앞), N1-otf vs N1w 부트스트랩 | `n1.json` |
| 7-3 | N2 vs N2w 부트스트랩 | `n1_N2.json` |
| 7-4 | 표 | `tables.md` |

- 부트스트랩은 각 run 의 기존 예측 csv(`pred_val`, `pred_test` = best 체크포인트)를 씁니다. 추론은 하지 않습니다.

In [ ]:
W1 = '--n2 N2w --diag W1_N2w --pp_from nad_N2_S01_s1'
!python -B diag_run.py n2 {W1} --combo a {D}

In [ ]:
!python -B diag_run.py n1 {W1} --n1a N1-otf --n1b N1w {D}

In [ ]:
!python -B diag_run.py n1 {W1} --n1a N2 --n1b N2w --n1_out n1_N2 {D}

In [ ]:
!python -B diag_run.py tables {W1} {D}
from IPython.display import Markdown, display
display(Markdown(open(f"{NAD_DIR}/diag/W1_N2w/tables.md", encoding='utf8').read()))